# **Option Pricing Library**

## **Overview**
This project implements vanilla and exotic options pricing and calculation of Greeks sensitivies from scratch. Where analytical solution for Greeks is not available, the finite difference method via bump and reprice is used.

The pricing and Greeks implementation spans European, American, Asian, Barrier, Binary options, and an autocallable. The methods used cover Black-Scholes analytics, Monte Carlo estimation with antithetic variance reduction, and CRR binomial tree.

In [3]:
import numpy as np
import yfinance as yf
from scipy.stats import norm
from datetime import datetime

### **1. European Options — Black-Scholes Analytical Pricing & Greeks**
The Black-Scholes model prices vanilla European options based on the assumption of Geometric Brownian Motion process for the underlying instrument (e.g., stock):

$$dS_t = (r - q)S_t dt + \sigma S_t dW_t$$

The closed form solution for the call and put price are:

$$C = Se^{-qT}N(d_1) - Ke^{-rT}N(d_2)$$
$$P = Ke^{-rT}N(-d_2) - Se^{-qT}N(-d_1)$$

where $$d_1 = \frac{\ln(S/K) + (r - q + \sigma^2/2)T}{\sigma\sqrt{T}},$$ 
$$d_2 = d_1 - \sigma\sqrt{T},$$
$N(\cdot)$ denotes the cumulative standard normal distribution function.$$

The Greeks represent an option's risk sensitivities:
- $\Delta$ (Delta) - the sensitivity of the **option's price** to a 1 unit move of the **underlying's price**.
- $\Gamma$ (Gamma) - the sensitivity of **$\Delta$** to a 1 unit move of the **underlying's price**.
- $\nu$ (Vega) - the sensitivity of the **option's price** to a 1 unit move of the **underlying's volatility**.
- $\Theta$ (Theta) - the sensitivity of the **option's price** to the **passage of time**.
- $\rho$ (Rho) - the sensitivity of the **option's price** to a 1 unit move of the **risk-free rate**.

In this part, Greeks are computed using closed-form Black-Scholes solutions.


In [4]:
def d1_d2(S, K, sigma, r, q, T):
    d1 = (np.log(S/K) + (r - q + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    return d1, d2


def black_scholes_pricer(S, K, sigma, r, q,  T, opt_type):

    if S <= 0 or K <= 0:
        raise ValueError("S and K must be greater than zero")
    
    if sigma <= 0:
        raise ValueError("sigma (volatility) must be greater than zero")
    
    if T <= 0:
        raise ValueError("T (time to option's expiry) must be greater than zero")
    
    if opt_type.lower() not in {"call", "put"}:
        raise ValueError("opt_type (option type) must be either 'call' or 'put'")

    d1, d2 = d1_d2(S, K, sigma, r, q, T)
    opt_type = opt_type.lower()

    if opt_type == "call":
        price = np.exp(-q * T) * S * norm.cdf(d1) - np.exp(-r * T) * K * norm.cdf(d2)
    elif opt_type == "put":
        price = np.exp(-r * T) * K * norm.cdf(-d2) - np.exp(-q * T) * S * norm.cdf(-d1)
    else:
        raise ValueError("Option type (opt_type) should be either 'call' or 'put'")
    
    return float(price)


def black_scholes_greeks(S, K, sigma, r, q, T, opt_type):

    if S <= 0 or K <= 0:
        raise ValueError("S and K must be greater than zero")
    
    if sigma <= 0:
        raise ValueError("sigma (volatility) must be greater than zero")
    
    if T <= 0:
        raise ValueError("T (time to option's expiry) must be greater than zero")
    
    if opt_type.lower() not in {"call", "put"}:
        raise ValueError("opt_type (option type) must be either 'call' or 'put'")
    
    d1, d2 = d1_d2(S, K, sigma, r, q, T)
    opt_type = opt_type.lower()

    gamma = (norm.pdf(d1) * np.exp(-q * T)) / (S * sigma * np.sqrt(T)) 
    vega = S * np.sqrt(T) * norm.pdf(d1) * np.exp(-q * T)
    vega_per_1pct_vol = vega / 100 # Dividing by 100 to represent the measure per single unit of vol (sensitivity to 1% increase in vol)

    if opt_type == "call":
        delta = np.exp(-q * T) * norm.cdf(d1) 

        theta = (
                -((S * norm.pdf(d1) * sigma * np.exp(-q * T))
                / (2 * np.sqrt(T))) 
                + q * S * norm.cdf(d1) * np.exp(-q * T) 
                - r * K * np.exp(-r * T) * norm.cdf(d2)
        )
        theta_daily = theta / 365 # convert to time decay per day

        rho = K * T * np.exp(-r * T) * norm.cdf(d2)
        rho_per_1pct_rate = rho / 100 # Dividing by 100 to represent the measure per 1 % move in int rates

    elif opt_type == "put":
        delta = np.exp(-q * T) * (norm.cdf(d1) - 1)

        theta = (
                -((S * norm.pdf(d1) * sigma * np.exp(-q * T)) 
                / (2 * np.sqrt(T))) 
                - q * S * norm.cdf(-d1) * np.exp(-q * T) 
                + r * K * np.exp(-r * T) * norm.cdf(-d2)
        )
        theta_daily = theta / 365
        
        rho = -K * T * np.exp(-r * T) * norm.cdf(-d2)
        rho_per_1pct_rate = rho / 100

    else:
        raise ValueError("Option type (opt_type) should be either 'call' or 'put'")


    greeks = {
        'delta': float(delta), 
        'gamma': float(gamma), 
        'vega' : float(vega_per_1pct_vol),
        'theta': float(theta_daily),
        'rho'  : float(rho_per_1pct_rate)
    }

    return greeks

### **2. Finite Difference Greeks via Bump & Reprice**
While vanilla European options have closed-form solutions for the Greeks, most other exotic derivatives do not. Fortunately, we can use finite difference method via bump and reprice to approximate the value of the Greeks with sufficient accuracy.

First, we can implement finite difference via bump and reprice using **forward method**:
$$\Delta = \frac{V(S + h) - V(S)}{h}$$
$$\Gamma = \frac{V(S + 2h) - 2V(S + h) + V(S)}{h^{2}}$$
$$\nu = \frac{V(\sigma + h) - V(\sigma)}{h}$$
$$\rho = \frac{V(r + h) - V(r)}{h}$$
Second, we can compute greeks using **central method**:
$$\Delta = \frac{V(S + h) - V(S - h)}{2h}$$
$$\Gamma = \frac{V(S + h) - 2V(S) + V(S - h)}{h^{2}}$$
$$\nu = \frac{V(\sigma + h) - V(\sigma - h)}{2h}$$
$$\rho = \frac{V(r + h) - V(r - h)}{2h}$$
Where:
- $V(\cdot)$ is the option's value / price
- $h$ is a small change for a given parameter

The central difference method is generally preferred as it achieves second-order accuracy $O(h^2)$ versus first-order accuracy $O(h)$ for the forward method meaning that the approximation error decreases faster as $h$ gets smaller.

For both methods, $\Theta$ is calculated as the daily price decay - the change in option's value over one trading day:
$$\Theta = V(T - dt) - V(T), \quad dt = \frac{1}{252}$$

In [5]:
def finite_diff_greeks(S, K, sigma, r, q, T, opt_type, ds=1e-4, dvol=1e-2, dr=1e-2, dt=1/252, method="central"):
    if S <= 0 or K <= 0:
        raise ValueError("S and K must be greater than zero")
    
    if sigma <= 0:
        raise ValueError("sigma (volatility) must be greater than zero")
    
    if T <= 0:
        raise ValueError("T (time to option's expiry) must be greater than zero")
    
    if opt_type.lower() not in {"call", "put"}:
        raise ValueError("opt_type (option type) must be either 'call' or 'put'")
    
    price_0 = black_scholes_pricer(S, K, sigma, r, q, T, opt_type)
    price_up_ds = black_scholes_pricer(S + ds, K, sigma, r, q, T, opt_type)
    
    # DELTA & GAMMA
    if method == "forward":
        delta = (price_up_ds - price_0) / ds
        price_up_2ds = black_scholes_pricer(S + 2*ds, K, sigma, r, q, T, opt_type)
        gamma = (price_up_2ds - 2 * price_up_ds + price_0) / (ds**2)


    elif method == "central":
        price_dn_ds = black_scholes_pricer(S - ds, K, sigma, r, q, T, opt_type)
        delta = (price_up_ds - price_dn_ds) / (2*ds)
        gamma = (price_up_ds - 2 * price_0 + price_dn_ds) / (ds**2)

    # VEGA
    price_up_vol = black_scholes_pricer(S, K, sigma + dvol, r, q, T, opt_type)
    if method == "forward":
        vega = (price_up_vol - price_0) / dvol
    elif method == "central":
        price_dn_vol = black_scholes_pricer(S, K, sigma - dvol, r, q, T, opt_type)
        vega = (price_up_vol - price_dn_vol) / (2 * dvol)
    vega_per_1pct_vol = vega * 0.01

    # THETA
    price_dn_T = black_scholes_pricer(S, K, sigma, r, q, T - dt, opt_type)
    theta = price_dn_T - price_0

    # RHO 
    price_up_r = black_scholes_pricer(S, K, sigma, r + dr, q, T, opt_type)
    if method == "forward":
        rho = (price_up_r - price_0) / dr
    elif method == "central":
        price_dn_r = black_scholes_pricer(S, K, sigma, r - dr, q, T, opt_type)
        rho = (price_up_r - price_dn_r) / (2 * dr)
    rho_per_1pct_rate = rho / 100

    return {
        "delta": float(delta),
        "gamma": float(gamma),
        "vega": float(vega_per_1pct_vol),
        "theta": float(theta),
        "rho": float(rho_per_1pct_rate)
    }

### **3. Monte Carlo Estimation with Antithetic Variate of Vanilla European Options**
Monte Carlo estimation is a numerical method that evaluates the option's price by simulating a large number of price paths for the underlying asset and averaging the payoffs. In this notebook, the method relies on the assumption that the underlying asset follows a Geometric Brownian Motion (GBM) process, where $S$ follows:
$$dS_t = (r - q)S_tdt + \sigma S_tdW_t$$
Where:
- $S_t$ - underlying asset price at time t
- $r$ - risk-free rate
- $q$ - continuous dividend yield
- $\sigma$ - volatility of the underlying asset
- $W_t$ - Wiener process at time t

The discretized solution for the GBM process is:
$$S_T = S_0\exp((r - q - \frac{\sigma^2}{2})T + \sigma \sqrt{T}Z)$$
Where:
- $Z$ - random sample from the standard normal distribution

The call and put option prices are calculated as follows:
$$C = e^{-rT}\frac{1}{N}\sum_{i=1}^{N}{\max(S_T^{(i)} - K, 0)}$$
$$P = e^{-rT}\frac{1}{N}\sum_{i=1}^{N}{\max(K - S_T^{(i)}, 0)}$$
Where:
- $N$ - number of simulations
- $K$ - option's strike price

To reduce the variance of the Monte Carlo estimator, antithetic variates are used. For each random draw $Z \sim \mathcal{N}(0,1)$, a paired draw $-Z$ is also used to generate negatively correlated path. The antithetic estimate for each pair is:
$$\hat{V}_i = \frac{V(Z_i) +V(-Z_i)}{2}$$

Since $V(Z_i)$ and $V(-Z_i)$ are negatively correlated, their average has lower variance than either individual estimate, reducing the standard error without requiring additional random number generation.

The precision of the estimate is measured by the standard error:
$$SE = \frac{\text{std(payoffs)}}{\sqrt{N/2}}$$

In [6]:
def monte_carlo_pricer_vanilla_eu(S, K, sigma, r, q, T, opt_type, num_iter=1_000_000, seed=42): # Antithetic: for variance reduction
    rng = np.random.default_rng(seed=seed)
    n = int(num_iter / 2)
    Z_pos = rng.standard_normal(size=n)
    Z_neg = Z_pos * -1

    S_T_pos = S * np.exp((r - q - sigma**2 * 0.5) * T + np.sqrt(T) * Z_pos * sigma)
    S_T_neg = S * np.exp((r - q - sigma**2 * 0.5) * T + np.sqrt(T) * Z_neg * sigma)

    if opt_type == "call":
        payoffs_pos = np.exp(-r * T) * np.maximum(S_T_pos - K, 0)
        payoffs_neg = np.exp(-r * T) * np.maximum(S_T_neg - K, 0)
    elif opt_type == "put":
        payoffs_pos = np.exp(-r * T) * np.maximum(K - S_T_pos, 0)
        payoffs_neg = np.exp(-r * T) * np.maximum(K - S_T_neg, 0)

    payoffs = (payoffs_pos + payoffs_neg) / 2

    price = np.mean(payoffs)
    se = np.std(payoffs, ddof=1) / np.sqrt(n)

    return price, se

### **4. Path-Dependent Antithetic Monte Carlo Pricer for Barrier Options with BGK Continuity Correction**

Barrier options are derivative instruments with a path-dependent payoff. Two main types of barrier options are **"knock-out"** and **"knock-in"** options. Knock-out options are those that become worthless if the underlying asset's price crosses a predetermined barrier level. Knock-in options are initially inactive until the underlying asset's price crosses the predetermined barrier, at which point they become standard options. These two types combined with the direction of the barrier give 4 barrier types: down-and-out, down-and-in, up-and-out, and up-and-in. 

Because the payoff depends on whether the barrier was breached during the option's life, closed-form solutions are limited and Monte Carlo is the standard approach for pricing barrier options.

Similarly to Monte Carlo estimation of vanilla options, we use antithetic variates for variance reduction, where for each draw $Z \sim \mathcal N(0,1)$, we also draw $-Z$ to generate a negatively correlated path. In contrast to vanilla options pricing, we need the full price path for each simulation to check whether the underlying asset has crossed the barrier. We do so by first simulating log returns:
$$R_{log} = (r - q - \frac{\sigma^2}{2})dt + \sigma \sqrt{dt} Z$$
Next, we build the full price path from the cumulative sum of log returns:
$$S_t = S_0\exp{(\sum_{i=1}^{t}R_{log}^{(i)})}$$

Barrier options are usually monitored on a discrete basis, and by default, the pricer models a discretely monitored barrier option which is checked at each simulation time step. When continuous=True, the Broadie-Glasserman-Kou continuity correction is applied with $\beta \approx 0.5826$, shifting the barrier toward the stock price to approximate a continuously monitored barrier:
$$B_{adj}^{up} = B \cdot e^{-\beta\sigma\sqrt{\Delta t}}$$
$$B_{adj}^{down} = B \cdot e^{+\beta\sigma\sqrt{\Delta t}}$$
Where:
- $B$ - initial barrier level.

This corrects for the fact that a discrete simulation underestimates barrier crossings relative to true continuous monitoring. As shown in Broadie, Glasserman & Kou (1997), "A Continuity Correction for Discrete Barrier Options", this adjustment significantly improves the accuracy of discrete approximations.

The price of a down-and-out call option is given by:
$$C_{DAO} = e^{-rT}\frac{1}{N}\sum_{i=1}^{N}\max(S_T^{(i)} - K,0) \cdot \mathbf{1}[\min_t S_t^{(i)} > B]$$
Where:
- $\mathbf{1}[\cdot]$ is the indicator function that equals 1 if the path never breached the barrier and 0 if it did.

The implementation covers all four barrier types — down-and-out, down-and-in, up-and-out, and up-and-in — for both calls and puts.

The calculation of standard error is done analogously as in the previous part.

In [7]:
def monte_carlo_barrier(S, K, B, sigma, r, q, T, opt_type, barrier_type, continuous=False, n_steps=252 , n_sims=1_000_000, seed=42):
    if barrier_type.lower() == "up-and-out" and S > B:
        return 0.0, 0.0
    elif barrier_type.lower() == "down-and-out" and S < B:
        return 0.0, 0.0
    
    rng = np.random.default_rng(seed=seed)

    Z_pos = rng.standard_normal((int(n_sims/2), n_steps))
    Z_neg = Z_pos * -1

    dt = T / n_steps

    log_returns_pos = (r - q - sigma**2 * 0.5) * dt + sigma * np.sqrt(dt) * Z_pos
    log_returns_neg = (r - q - sigma**2 * 0.5) * dt + sigma * np.sqrt(dt) * Z_neg

    S_T_pos = np.exp(np.log(S) + np.cumsum(log_returns_pos, axis=1))
    S_T_neg = np.exp(np.log(S) + np.cumsum(log_returns_neg, axis=1))

    if continuous:
        beta = 0.5826
        if "up" in barrier_type.lower():
            B_adj = B * np.exp(-beta * sigma * np.sqrt(dt))
        elif "down" in barrier_type.lower():
            B_adj = B * np.exp(+beta * sigma * np.sqrt(dt))
    else:
        B_adj = B

    if barrier_type.lower() == "up-and-out":
        barrier_breached_pos = np.any(S_T_pos >= B_adj, axis=1)
        barrier_breached_neg = np.any(S_T_neg >= B_adj, axis=1)
        if opt_type.lower() == "call":
            payoffs_pos = np.exp(-r * T) * (np.maximum(S_T_pos[:, -1] - K, 0)) * ~barrier_breached_pos  
            payoffs_neg = np.exp(-r * T) * (np.maximum(S_T_neg[:, -1] - K, 0)) * ~barrier_breached_neg
        elif opt_type.lower() == "put":
            payoffs_pos = np.exp(-r * T) * (np.maximum(K - S_T_pos[:, -1], 0)) * ~barrier_breached_pos
            payoffs_neg = np.exp(-r * T) * (np.maximum(K - S_T_neg[:, -1], 0)) * ~barrier_breached_neg
    elif barrier_type.lower() == "up-and-in":
        barrier_breached_pos = np.any(S_T_pos >= B_adj, axis=1)
        barrier_breached_neg = np.any(S_T_neg >= B_adj, axis=1)
        if opt_type.lower() == "call":
            payoffs_pos = np.exp(-r * T) * (np.maximum(S_T_pos[:, -1] - K, 0)) * barrier_breached_pos  
            payoffs_neg = np.exp(-r * T) * (np.maximum(S_T_neg[:, -1] - K, 0)) * barrier_breached_neg
        elif opt_type.lower() == "put":
            payoffs_pos = np.exp(-r * T) * (np.maximum(K - S_T_pos[:, -1], 0)) * barrier_breached_pos
            payoffs_neg = np.exp(-r * T) * (np.maximum(K - S_T_neg[:, -1], 0)) * barrier_breached_neg
    elif barrier_type.lower() == "down-and-out":
        barrier_breached_pos = np.any(S_T_pos <= B_adj, axis=1)
        barrier_breached_neg = np.any(S_T_neg <= B_adj, axis=1)
        if opt_type.lower() == "call":
            payoffs_pos = np.exp(-r * T) * (np.maximum(S_T_pos[:, -1] - K, 0)) * ~barrier_breached_pos  
            payoffs_neg = np.exp(-r * T) * (np.maximum(S_T_neg[:, -1] - K, 0)) * ~barrier_breached_neg
        elif opt_type.lower() == "put":
            payoffs_pos = np.exp(-r * T) * (np.maximum(K - S_T_pos[:, -1], 0)) * ~barrier_breached_pos
            payoffs_neg = np.exp(-r * T) * (np.maximum(K - S_T_neg[:, -1], 0)) * ~barrier_breached_neg
    elif barrier_type.lower() == "down-and-in":
        barrier_breached_pos = np.any(S_T_pos <= B_adj, axis=1)
        barrier_breached_neg = np.any(S_T_neg <= B_adj, axis=1)
        if opt_type.lower() == "call":
            payoffs_pos = np.exp(-r * T) * (np.maximum(S_T_pos[:, -1] - K, 0)) * barrier_breached_pos  
            payoffs_neg = np.exp(-r * T) * (np.maximum(S_T_neg[:, -1] - K, 0)) * barrier_breached_neg
        elif opt_type.lower() == "put":
            payoffs_pos = np.exp(-r * T) * (np.maximum(K - S_T_pos[:, -1], 0)) * barrier_breached_pos
            payoffs_neg = np.exp(-r * T) * (np.maximum(K - S_T_neg[:, -1], 0)) * barrier_breached_neg
    
    payoffs = (payoffs_pos + payoffs_neg) / 2
    price = np.mean(payoffs)
    se = np.std(payoffs) / np.sqrt(int(n_sims)/2)

    return price, se

In [8]:
# 5. Finite Difference Greeks for a Barrier Option via Bump and Revalue (B & R) - Analagously as in 2.
def finite_diff_greeks_barrier(S, K, B, sigma, r, q, T, opt_type, barrier_type, ds=1e-2, dvol=1e-4, dr=1e-4, dt=1/252, method="central", seed=42):

    def get_p(S_bump, sigma_bump, r_bump, T_bump):
        return monte_carlo_barrier(S_bump, K, B, sigma_bump, r_bump, q, T_bump, opt_type, barrier_type, seed=seed)[0]
    # DELTA & GAMMA
    price_0 = get_p(S, sigma, r, T)
    price_up_ds = get_p(S + ds, sigma, r, T)

    if method == "forward":
        price_up_2ds = get_p(S + 2*ds, sigma, r, T)
        delta = (price_up_ds - price_0) / ds
        gamma = (price_up_2ds - 2 * price_up_ds + price_0) / (ds**2)
    elif method == "central":
        price_dn_ds = get_p(S - ds, sigma, r, T)
        delta = (price_up_ds - price_dn_ds) / (2 * ds)
        gamma = (price_up_ds - 2 * price_0 + price_dn_ds) / (ds**2)

    # VEGA
    price_up_dvol = get_p(S, sigma + dvol, r, T)
    if method == "forward":
        vega = (price_up_dvol - price_0) / dvol
    elif method == "central":
        price_dn_dvol = get_p(S, sigma - dvol, r, T)
        vega = (price_up_dvol - price_dn_dvol) / (2 * dvol)
    vega_per_1pct_vol = vega * 0.01

    # THETA
    if T - dt <= 0:
        theta_daily = 0.0 
    else:
        price_dn_dt = get_p(S, sigma, r, T - dt)
        theta_daily = price_dn_dt - price_0

    # RHO
    price_up_dr = get_p(S, sigma, r + dr, T)
    if method == "forward":
        rho = (price_up_dr - price_0) / dr
    elif method == "central":
        price_dn_dr = get_p(S, sigma, r - dr, T)
        rho = (price_up_dr - price_dn_dr) / (2 * dr)
    rho_per_1pct_rate = rho / 100

    return {
        "delta": delta,
        "gamma": gamma,
        "vega":  vega_per_1pct_vol,
        "theta": theta_daily,
        "rho":   rho_per_1pct_rate
    }

### **6. Path Dependent Antithetic Monte Carlo Pricer for Arithmetic Asian Option**

Similarly to barrier options, Asian options are derivatives with a path-dependent payoff. Their payoff depends on the average price reached during the option's life. Analogously to the way barrier option was priced, this Monte Carlo pricer uses antithetic variates for variance reduction. Next, log returns and full price paths are computed. Payoff of the Asian call and put are defined as:
$$\text{Payoff}_{Ci} = e^{-rT}\max(S_{ave}^{(i)} - K,0)$$
$$\text{Payoff}_{Pi} = e^{-rT}\max(K - S_{ave}^{(i)},0)$$
where $S_{ave}^{(i)}$ is the arithmetic average price of the underlying along the $i$-th simulated path:
$$S_{ave}^{(i)} = \frac{1}{n}\sum_{i=1}^{n}S_t^{(i)}$$
Final Asian option price is derived as:
$$V = \frac{1}{N}\sum_{i=1}^{N}\text{Payoff}_i$$

The standard error is computed analogously as in previous parts.

In [9]:
def monte_carlo_asian(S, K, sigma, r, q, T, opt_type, n_steps=252, n_sims=1_000_000, seed=42):
    rng = np.random.default_rng(seed=seed)

    Z_pos = rng.standard_normal((int(n_sims/2), n_steps))
    Z_neg = Z_pos * -1

    dt = T / n_steps

    log_returns_pos = (r - q - sigma**2 * 0.5) * dt + sigma * np.sqrt(dt) * Z_pos
    log_returns_neg = (r - q - sigma**2 * 0.5) * dt + sigma * np.sqrt(dt) * Z_neg

    S_T_pos = np.exp((np.log(S) + np.cumsum(log_returns_pos, axis=1)))
    S_T_neg = np.exp((np.log(S) + np.cumsum(log_returns_neg, axis=1)))

    S_0_col = np.full((int(n_sims/2), 1), S)

    S_T_pos = np.hstack([S_0_col, S_T_pos])
    S_T_neg = np.hstack([S_0_col, S_T_neg])

    if opt_type.lower() == "call":
        payoffs_pos = np.exp(-r * T) * np.maximum(np.mean(S_T_pos, axis=1) - K, 0)
        payoffs_neg = np.exp(-r * T) * np.maximum(np.mean(S_T_neg, axis=1) - K, 0)
    elif opt_type.lower() == "put":
        payoffs_pos = np.exp(-r * T) * np.maximum(K - np.mean(S_T_pos, axis=1), 0)
        payoffs_neg = np.exp(-r * T) * np.maximum(K - np.mean(S_T_neg, axis=1), 0)

    payoffs = (payoffs_pos + payoffs_neg) / 2
    
    price = np.mean(payoffs)
    se = np.std(payoffs) / np.sqrt(int(n_sims/2))

    return price, se

# Greeks analagously as in 2.
def finite_diff_greeks_asian(S, K, sigma, r, q, T, opt_type, ds=1e-2, dvol=1e-4, dr=1e-4, dt=1/252, method="central", seed=42): 
    def get_price(S_bump, sigma_bump, r_bump, T_bump):
        return monte_carlo_asian(S_bump, K, sigma_bump, r_bump, q, T_bump, opt_type, seed=seed)[0]
    price_0 = get_price(S, sigma, r, T)
    price_up_ds = get_price(S + ds, sigma, r, T)

    # DELTA & GAMMA
    if method == "forward":
        price_up_2ds = get_price(S + 2*ds, sigma, r, T)
        delta = (price_up_ds - price_0) / ds
        gamma = (price_up_2ds - 2 * price_up_ds + price_0) / (ds**2)
    elif method == "central":
        price_dn_ds = get_price(S - ds, sigma, r, T)
        delta = (price_up_ds - price_dn_ds) / (2 * ds)
        gamma = (price_up_ds - 2 * price_0 + price_dn_ds) / (ds**2)

    # VEGA
    price_up_dvol = get_price(S, sigma + dvol, r, T)
    if method == "forward":
        vega = (price_up_dvol - price_0) / (dvol)
    elif method == "central":
        price_dn_dvol = get_price(S, sigma - dvol, r, T)
        vega = (price_up_dvol - price_dn_dvol) / (2 * dvol)
    vega_per_1pct_vol = vega * 0.01

    # THETA
    if T - dt <= 0 :
        theta_daily = 0.0
    else:
        price_dn_dt = get_price(S, sigma, r, T - dt)
        theta_daily = price_dn_dt - price_0

    # RHO
    price_up_dr = get_price(S, sigma, r + dr, T)
    if method == "forward":
        rho = (price_up_dr - price_0) / dr
    elif method == "central":
        price_dn_dr = get_price(S, sigma, r - dr, T)
        rho = (price_up_dr - price_dn_dr) / (2 * dr)
    rho_per_1pct_rate = rho / 100

    return {
        "delta": delta,
        "gamma": gamma,
        "vega":  vega_per_1pct_vol,
        "theta": theta_daily,
        "rho":   rho_per_1pct_rate
    }

### **7. Pricing Plain Vanilla American Option using Binomial Tree Approach**

American options can be exercised on any trading day, while European ones can be exercised only at the end of their lives. This is an important caveat in the pricing algorithm, because at each time step we must check whether it is more valuable to exercise the option or hold it. There are no closed-forms solutions in general for the price of an American option, and we also cannot use standard Monte Carlo approach because it does not allow for the *backward induction* required for pricing American options.

The standard approach for pricing American options is the binomial tree method developed by Cox, Ross, and Rubinstein in 1979. The essence of this technique is that at each time step the underlying asset price can move either up or down by a certain factor with certain probabilities, which forms the tree structure. In the standard CRR model:
- $u = e^{\sigma\sqrt{\Delta t}}$ - up move factor
- $d = 1/u$ - down move factor
- $a = e^{(r - q)dt}$ - "growth" factor
- $p = \frac{a - d}{u - d}$ - probability of the up move

These parametres are used to build the stock tree:
$$ S_{i, j} = S_0 \cdot u^{j - i} \cdot d^{i}$$
where:
- $j$ - time step (column in the tree)
- $i$ - number of down moves at that node
- $j - i$ - number of up moves at that node

At expiry, the option value equals the intrinsic value $(C_T = \max(S_T - K,0))$. Working backward through the tree, at each node the option value is:
$$V_{i, j} = \max\left(e^{-r\Delta t}[p \cdot V_{i, j+1} + (1-p) \cdot V_{i+1,j+1}],\ E_{i,j}\right)$$
where $E_{i,j}$ is the exercise value at that node. Taking the maximum of continuation and exercise value at each node is what captures the American early exercise feature.
Finally, the option's price is given by $V_{0,0}$.

In [10]:
def binomial_tree_amer_opt(S, K, sigma, r, q, T, opt_type, n_steps=500):
    dt = T / n_steps

    u = np.exp(sigma * np.sqrt(dt))
    d = 1 / u
    a = np.exp((r - q) * dt)
    p = (a - d) / (u - d)

    stock_tree = np.zeros((n_steps + 1, n_steps + 1))
    option_tree = np.zeros((n_steps + 1, n_steps + 1))

    for j in range(n_steps + 1):
        for i in range(j + 1):
            stock_tree[i, j] = S * u**(j - i) * d**(i)

    if opt_type.lower() == "call":
        option_tree[:, n_steps] = np.maximum(stock_tree[:, n_steps] - K, 0)
    elif opt_type.lower() == "put":
        option_tree[:, n_steps] = np.maximum(K - stock_tree[:, n_steps], 0)

    for j in range(n_steps - 1, -1, -1):
        for i in range(j + 1):
            continuation_value = np.exp(-r * dt) * (p * option_tree[i, j + 1] + (1 - p) * option_tree[i + 1, j + 1])
            if opt_type.lower() == "call":
                exercise_value = np.maximum(stock_tree[i, j] - K, 0)
            elif opt_type.lower() == "put":
                exercise_value = np.maximum(K - stock_tree[i, j], 0)
            option_tree[i, j] = np.maximum(continuation_value, exercise_value)

    return option_tree[0, 0]


# Greeks analagously as in 2.
def finite_diff_greeks_american(S, K, sigma, r, q, T, opt_type, ds=1e-2, dvol=1e-4, dr=1e-4, dt=1/252, method="central"):
    def get_price(S_bump, sigma_bump, r_bump, T_bump):
        return binomial_tree_amer_opt(S_bump, K, sigma_bump, r_bump, q, T_bump, opt_type)
    price_0 = get_price(S, sigma, r, T)
    price_up_ds = get_price(S + ds, sigma, r, T)

    # DELTA & GAMMA
    if method == "forward":
        price_up_2ds = get_price(S + 2*ds, sigma, r, T)
        delta = (price_up_ds - price_0) / ds
        gamma = (price_up_2ds - 2 * price_up_ds + price_0) / (ds**2)
    elif method == "central":
        price_dn_ds = get_price(S - ds, sigma, r, T)
        delta = (price_up_ds - price_dn_ds) / (2 * ds)
        gamma = (price_up_ds - 2 * price_0 + price_dn_ds) / (ds**2)
    
    # VEGA
    price_up_dvol = get_price(S, sigma + dvol, r, T)
    if method == "forward":
        vega = (price_up_dvol - price_0) / dvol
    elif method == "central":
        price_dn_dvol = get_price(S, sigma - dvol, r, T)
        vega = (price_up_dvol - price_dn_dvol) / (2 * dvol)
    vega_per_1pct_change = vega * 0.01

    # THETA
    price_dn_dt = get_price(S, sigma, r, T - dt)
    if T - dt <= 0:
        theta_daily = 0.0
    else:
        theta_daily = price_dn_dt - price_0

    # RHO
    price_up_dr = get_price(S, sigma, r + dr, T)
    if method == "forward":
        rho = (price_up_dr - price_0) / (dr)
    elif method == "central":
        price_dn_dr = get_price(S, sigma, r - dr, T)
        rho = (price_up_dr - price_dn_dr) / (2 * dr)
    rho_per_1pct_change = rho / 100

    return {
        "delta": delta,
        "gamma": gamma,
        "vega": vega_per_1pct_change,
        "theta": theta_daily,
        "rho": rho_per_1pct_change
    }

### **8. Autocallable Pricer**
An autocallable is a structured product that can automatically mature early and pay out a coupon together with the initial principal investment if the underlying asset reaches the predefined *autocall* barrier. This monitoring happens on the predefined observation dates. 

If the product autocalls on observation date $t$, the investor receives the principal plus a coupon proportional to the number of periods elapsed:
$$\text{Payoff}_{autocall} = e^{-rT} \cdot S_0 \cdot (1 + c \cdot t)$$
where $c$ is the annual coupon rate and $t$ is the time of the autocall in years.

If the autocallable matures without reaching the autocall barrier, there are 2 scenarios:

1. The underlying is *above* the *protection* barrier: the investor is returned the the full principal investment back at maturity.
2. The underlying is *below* the *protection* barrier: the investor is returned the market value of the underlying, resulting in a loss proportional to the decline below the protection barrier.

$$\text{Payoff}_{maturity} = e^{-rT} \cdot \begin{cases}
S_0 & \text{if } S_T \geq B_{protection} \\
S_T & \text{if } S_T < B_{protection}
\end{cases}$$



In [11]:
def pricer_autocallable(S, autocall_barrier, protection_barrier, coupon_rate, sigma, r, q, T, 
                        obs_indices=[252,504,756], n_steps=756, n_sims=100_000, seed=42):
    rng = np.random.default_rng(seed=seed)

    Z_pos = rng.standard_normal((int(n_sims/2), n_steps))
    Z_neg = Z_pos * -1

    dt = T / n_steps

    log_returns_pos = (r - q - sigma**2 * 0.5) * dt + sigma * np.sqrt(dt) * Z_pos
    log_returns_neg = (r - q - sigma**2 * 0.5) * dt + sigma * np.sqrt(dt) * Z_neg

    paths_pos = np.exp(np.log(S) + np.cumsum(log_returns_pos, axis=1))
    paths_neg = np.exp(np.log(S) + np.cumsum(log_returns_neg, axis=1))

    S_0_col = np.full((int(n_sims / 2), 1), S)

    paths_pos = np.hstack([S_0_col, paths_pos])
    paths_neg = np.hstack([S_0_col, paths_neg])
    paths_all = np.vstack((paths_pos, paths_neg))

    payoffs = np.zeros(n_sims)

    for i in range(n_sims):
        path = paths_all[i]
        autocalled = False

        for year, idx in enumerate(obs_indices):
            act_year = year + 1

            if path[idx] >= autocall_barrier:
                payoff = np.exp(-r * act_year) * (S * (1 + (coupon_rate * act_year)))
                payoffs[i] = payoff
                autocalled = True
                break
        
        if not autocalled:
            final_price = path[-1]

            if final_price >= protection_barrier:
                payoffs[i] = np.exp(-r * T) * S
            else:
                payoffs[i] = np.exp(-r * T) * final_price

    half = int(n_sims / 2)
    paired_payoffs = (payoffs[:half] + payoffs[half:]) / 2

    price = np.mean(paired_payoffs)
    se = np.std(paired_payoffs, ddof=1) / np.sqrt(half)

    return price, se

# Greeks analagously as in 2.
def finite_diff_greeks_autocallable(S, autocall_barrier, protection_barrier, coupon_rate, sigma, r, q, T,
                                    obs_indices=[252,504,756], ds=1e-2, dvol=1e-4, dr=1e-4, dt=1/252, method="central", seed=42):
    def get_price(S_bump, sigma_bump, r_bump, T_bump):
        return pricer_autocallable(S_bump, autocall_barrier, protection_barrier, coupon_rate, sigma_bump, r_bump, q, T_bump, obs_indices, seed=seed)[0]
    
    price_0 = get_price(S, sigma, r, T)

    # DELTA & GAMMA
    price_up_ds = get_price(S + ds, sigma, r, T)
    if method == "forward":
        price_up_2ds = get_price(S + 2*ds, sigma, r, T)
        delta = (price_up_ds - price_0) / (ds)
        gamma = (price_up_2ds - 2 * price_up_ds + price_0) / (ds**2)
    elif method == "central":
        price_dn_ds = get_price(S - ds, sigma, r, T)
        delta = (price_up_ds - price_dn_ds) / (2*ds)
        gamma = (price_up_ds - 2 * price_0 + price_dn_ds) / (ds**2)

    # VEGA
    price_up_dvol = get_price(S, sigma + dvol, r, T)
    if method == "forward":
        vega = (price_up_dvol - price_0) / dvol
    elif method == "central":
        price_dn_dvol = get_price(S, sigma - dvol, r, T)
        vega = (price_up_dvol - price_dn_dvol) / (2*dvol)
    vega_per_1pct_change = vega * 0.01

    # THETA
    price_dn_dt = get_price(S, sigma, r, T - dt)
    theta = price_dn_dt - price_0

    # RHO
    price_up_dr = get_price(S, sigma, r + dr, T)
    if method == "forward":
        rho = (price_up_dr - price_0) / dr
    elif method == "central":
        price_dn_dr = get_price(S, sigma, r - dr, T)
        rho = (price_up_dr - price_dn_dr) / (2*dr)
    rho_per_1pct_change = rho / 100

    return {
        "delta": delta,
        "gamma": gamma,
        "vega": vega_per_1pct_change,
        "theta": theta,
        "rho": rho_per_1pct_change
    }    

### **9. Binary Options Pricer**
Binary options or cash-or-nothing options are derivatives that pay pay a fixed amount $Q$ if the underlying finishes above (call) or below (put) the strike at expiry. Those options can be priced either using Black-Scholes analytical approach or Monte Carlo estimation:
- **Black-Scholes**:
$$C = Q \cdot e^{-rT} \cdot \mathcal{N}(d_2)$$
$$P = Q \cdot e^{-rT} \cdot \mathcal{N}(-d_2)$$
- **Monte-Carlo**:
$$\text{Payoff}_{call}^{(i)} = e^{-rT} \cdot Q \cdot \mathbf{1}[S_T^{(i)} > K]$$
$$\text{Payoff}_{put}^{(i)} = e^{-rT} \cdot Q \cdot \mathbf{1}[S_T^{(i)} < K]$$
$$\text{Price} = \frac {1}{N} \sum_{i=1}^{N} \text{Payoff}_{i}$$
$$\text{SE} = \frac{\text{std}(\text{payoffs})}{\sqrt{N/2}}$$

In [12]:
def binary_option_analytical(S, Q, K, sigma, r, q, T, opt_type):
    d2 = d1_d2(S, K, sigma, r, q, T)[1]
    if opt_type.lower() == "call":
        price = Q * np.exp(-r * T) * norm.cdf(d2)
    elif opt_type.lower() == "put":
        price = Q * np.exp(-r * T) * norm.cdf(-d2)
    return price

def binary_option_monte_carlo(S, Q, K, sigma, r, q, T, opt_type, num_iter=1_000_000, seed=42):
    rng = np.random.default_rng(seed=seed)
    n = int(num_iter / 2)
    Z_pos = rng.standard_normal(size=n)
    Z_neg = Z_pos * -1

    S_T_pos = S * np.exp((r - q - sigma**2 * 0.5) * T + np.sqrt(T) * Z_pos * sigma)
    S_T_neg = S * np.exp((r - q - sigma**2 * 0.5) * T + np.sqrt(T) * Z_neg * sigma)

    if opt_type.lower() == "call":
        payoffs_pos = np.exp(-r * T) * np.where(S_T_pos > K, Q, 0)
        payoffs_neg = np.exp(-r * T) * np.where(S_T_neg > K, Q, 0)
    elif opt_type.lower() == "put":
        payoffs_pos = np.exp(-r * T) * np.where(S_T_pos < K, Q, 0)
        payoffs_neg = np.exp(-r * T) * np.where(S_T_neg < K, Q, 0)

    payoffs = (payoffs_pos + payoffs_neg) / 2

    price = np.mean(payoffs)
    se = np.std(payoffs, ddof=1) / np.sqrt(n)

    return price, se

### **10. OOP Refactoring**

All pricing functions are wrapped in an object-oriented class hierarchy for clean and intuitive usage. The base `Option` class stores shared parameters (S, K, σ, r, q, T, opt_type). Subclasses inherit these and expose `price()` and `greeks()` methods backed by the appropriate pricing functions.

| Class | Pricing Method | Greeks Method |
|---|---|---|
| EuropeanOption | Black-Scholes + Monte Carlo | Analytical BS |
| BarrierOption | Monte Carlo + BGK | Finite difference (B & R)|
| AsianOption | Monte Carlo | Finite difference (B & R)|
| AmericanOption | Binomial tree (CRR) | Finite difference (B & R)|
| BinaryOption | Analytical + Monte Carlo | — |
| Autocallable | Monte Carlo | Finite difference (B & R)|

In [13]:
class Option():
    def __init__(self, S, K, sigma, r, q, T, opt_type):
        self.S = S
        self.K = K
        self.sigma = sigma
        self.r = r
        self.q = q
        self.T = T
        self.opt_type = opt_type.lower()


class EuropeanOption(Option):
    def price_analytical(self):
        return black_scholes_pricer(self.S, self.K, self.sigma, self.r, self.q, self.T, self.opt_type)
    def price_monte_carlo(self):
        return monte_carlo_pricer_vanilla_eu(self.S, self.K, self.sigma, self.r, self.q, self.T, self.opt_type)
    def greeks(self):
        return black_scholes_greeks(self.S, self.K, self.sigma, self.r, self.q, self.T, self.opt_type)


class BarrierOption(Option):
    def __init__(self, S, K, B, sigma, r, q, T, opt_type, barrier_type):
        super().__init__(S, K, sigma, r, q, T, opt_type)
        self.B = B
        self.barrier_type = barrier_type.lower()
    def price(self):
        return monte_carlo_barrier(self.S, self.K, self.B, self.sigma, self.r, self.q, self.T, self.opt_type, self.barrier_type)
    def greeks(self):
        return finite_diff_greeks_barrier(self.S, self.K, self.B, self.sigma, self.r, self.q, self.T, self.opt_type, self.barrier_type)
    

class AsianOption(Option):
    def price(self):
        return monte_carlo_asian(self.S, self.K, self.sigma, self.r, self.q, self.T, self.opt_type)
    def greeks(self):
        return finite_diff_greeks_asian(self.S, self.K, self.sigma, self.r, self.q, self.T, self.opt_type)
    
    
class AmericanOption(Option):
    def price(self):
        return binomial_tree_amer_opt(self.S, self.K, self.sigma, self.r, self.q, self.T, self.opt_type)
    def greeks(self):
        return finite_diff_greeks_american(self.S, self.K, self.sigma, self.r, self.q, self.T, self.opt_type)

class BinaryOption(Option):
    def __init__(self, S, Q, K, sigma, r, q, T, opt_type):
        super().__init__(S, K, sigma, r, q, T, opt_type)
        self.Q = Q
    def price_analytical(self):
        return binary_option_analytical(self.S, self.Q, self.K, self.sigma, self.r, self.q, self.T, self.opt_type)
    def price_monte_carlo(self):
        return binary_option_monte_carlo(self.S, self.Q,  self.K, self.sigma, self.r, self.q, self.T, self.opt_type)
    
class Autocallable():
    def __init__(self, S, autocall_barrier, protection_barrier, coupon_rate, sigma, r, q, T, obs_indices):
        self.S = S
        self.autocall_barrier = autocall_barrier
        self.protection_barrier = protection_barrier
        self.coupon_rate = coupon_rate
        self.sigma = sigma
        self.r = r
        self.q = q
        self.T = T
        self.obs_indices = obs_indices
        
    def price(self):
        return pricer_autocallable(self.S, self.autocall_barrier, self.protection_barrier, self.coupon_rate, 
                                   self.sigma, self.r, self.q, self.T, self.obs_indices)
    def greeks(self):
        return finite_diff_greeks_autocallable(self.S, self.autocall_barrier, self.protection_barrier, self.coupon_rate, 
                                   self.sigma, self.r, self.q, self.T, self.obs_indices)

### **11. Live Market Data Integration**
The `build_option_from_ticker` function fetches live market data via `yfinance` to automatically populate option parameters:

- **S** — latest closing price from 1-year price history
- **σ** — implied volatility from the closest available strike 
  in the options chain
- **r** — risk-free rate from the 13-week US Treasury bill yield (^IRX)
- **q** — continuous dividend yield from stock metadata
- **T** — time to expiry in years, computed from the expiration date

The function accepts any option class from the OOP hierarchy as 
the `option_class` parameter, defaulting to `EuropeanOption`.

In [14]:
# 11. Fetching live market data and initialising an Option object
def build_option_from_ticker(ticker, K, expiration_date, opt_type, option_class = EuropeanOption):
    stock = yf.Ticker(ticker)

    hist = stock.history(period="1y")
    S = hist["Close"].iloc[-1]

    chain = stock.option_chain(expiration_date)
    options_df = chain.calls if opt_type.lower() == "call" else chain.puts

    closest_idx = (options_df['strike'] - K).abs().argmin()
    target_option = options_df.iloc[closest_idx]

    actual_K = target_option["strike"]
    sigma = target_option["impliedVolatility"]

    exp_dt = datetime.strptime(expiration_date, "%Y-%m-%d")
    days_to_expiry = (exp_dt - datetime.now()).days
    T = max(days_to_expiry / 365.0, 1e-5)

    q = stock.info.get("dividendYield", 0.0)
    if q is None: q = 0.0

    treasury = yf.Ticker("^IRX")
    r = treasury.history(period="1d")["Close"].iloc[-1] / 100.0

    return option_class(S=S, K=actual_K, sigma=sigma, r=r, q=q, T=T, opt_type=opt_type)

### **12. Demo Usage**

In [16]:
print("=== 1. LIVE MARKET DATA: AAPL European Call ===")
try:
    aapl_call = build_option_from_ticker(
        ticker="AAPL", 
        K=185.0, 
        expiration_date="2026-06-18", 
        opt_type="call", 
        option_class=EuropeanOption
    )
    
    print(f"Fetched Parameters:")
    print(f"  Underlying (S): ${aapl_call.S:.2f}")
    print(f"  Implied Vol (σ): {aapl_call.sigma:.2%}")
    print(f"  Risk-Free Rate (r): {aapl_call.r:.2%}")
    print(f"  Time to Expiry (T): {aapl_call.T:.4f} years\n")
    
    print(f"Pricing:")
    print(f"  Black-Scholes: ${aapl_call.price_analytical():.4f}")
    mc_price, mc_se = aapl_call.price_monte_carlo()
    print(f"  Monte Carlo:   ${mc_price:.4f} (SE: {mc_se:.4f})\n")
    
    print("Greeks:")
    for greek, value in aapl_call.greeks().items():
        print(f"  {greek.capitalize()}: {value:.4f}")
        
except Exception as e:
    print(f"Error fetching live data. Check ticker and expiration date.\nException: {e}")


print("\n" + "="*47)
print("=== 2. EXOTIC PRICING: Down-and-Out Put ===")
barrier_put = BarrierOption(
    S=100.0, K=100.0, B=90.0, 
    sigma=0.20, r=0.05, q=0.0, T=1.0, 
    opt_type="put", barrier_type="down-and-out"
)

b_price, b_se = barrier_put.price()
print(f"Monte Carlo Price: ${b_price:.4f} (SE: {b_se:.4f})")

print("Finite Difference Greeks:")
for greek, value in barrier_put.greeks().items():
    print(f"  {greek.capitalize()}: {value:.4f}")


print("\n" + "="*47)
print("=== 3. AMERICAN PRICING: CRR Binomial Tree ===")
american_call = AmericanOption(
    S=50.0, K=50.0, sigma=0.30, r=0.04, q=0.02, T=0.5, opt_type="call"
)

print(f"Binomial Tree Price: ${american_call.price():.4f}")

print("Finite Difference Greeks:")
for greek, value in american_call.greeks().items():
    print(f"  {greek.capitalize()}: {value:.4f}")

=== 1. LIVE MARKET DATA: AAPL European Call ===
Fetched Parameters:
  Underlying (S): $308.82
  Implied Vol (σ): 100.24%
  Risk-Free Rate (r): 3.59%
  Time to Expiry (T): 0.0685 years

Pricing:
  Black-Scholes: $117.6908
  Monte Carlo:   $117.7111 (SE: 0.0248)

Greeks:
  Delta: 0.9542
  Gamma: 0.0006
  Vega: 0.0424
  Theta: 0.1802
  Rho: 0.1212

=== 2. EXOTIC PRICING: Down-and-Out Put ===
Monte Carlo Price: $0.1899 (SE: 0.0009)
Finite Difference Greeks:
  Delta: 0.0133
  Gamma: -1.0332
  Vega: -0.0234
  Theta: 0.0008
  Rho: -0.0061

=== 3. AMERICAN PRICING: CRR Binomial Tree ===
Binomial Tree Price: $4.4095
Finite Difference Greeks:
  Delta: 0.5553
  Gamma: 3.4898
  Vega: 0.1379
  Theta: -0.0180
  Rho: 0.1168
